In [1]:
from google.colab import files

uploaded = files.upload()

Saving olist_orders_dataset.csv to olist_orders_dataset.csv
Saving olist_products_dataset.csv to olist_products_dataset.csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset.csv
Saving product_category_name_translation.csv to product_category_name_translation.csv
Saving olist_geolocation_dataset.csv to olist_geolocation_dataset.csv
Saving olist_order_items_dataset.csv to olist_order_items_dataset.csv
Saving olist_order_payments_dataset.csv to olist_order_payments_dataset.csv
Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset.csv
Saving olist_customers_dataset.csv to olist_customers_dataset.csv


In [12]:
import pandas as pd
import io

tables = {}

for filename, filedata in uploaded.items():
    table_name = filename.replace(".csv", "")

    tables[table_name] = pd.read_csv(
        io.BytesIO(filedata)
    )

print("Loaded tables:")
for name, df in tables.items():
    print(name, "→", df.shape)

Loaded tables:
olist_orders_dataset → (99441, 8)
olist_products_dataset → (32951, 9)
olist_sellers_dataset → (3095, 4)
product_category_name_translation → (71, 2)
olist_geolocation_dataset → (1000163, 5)
olist_order_items_dataset → (112650, 7)
olist_order_payments_dataset → (103886, 5)
olist_order_reviews_dataset → (99224, 7)
olist_customers_dataset → (99441, 5)


In [115]:
from google.colab import files

files.download("/content/cleaned_olist.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [14]:
import pandas as pd

# ==========================================================
# 1. TABLE-SPECIFIC BUSINESS RULES
# ==========================================================

rules = {

    "olist_order_reviews_dataset": {
        "review_score": lambda x: x.between(1, 5)
    },

    "olist_order_payments_dataset": {
        "payment_value": lambda x: x >= 0,
        "payment_installments": lambda x: x >= 1,
        "payment_sequential": lambda x: x >= 1
    },

    "olist_order_items_dataset": {
        "price": lambda x: x >= 0,
        "freight_value": lambda x: x >= 0,
        "order_item_id": lambda x: x >= 1
    },

    "olist_products_dataset": {
        "product_weight_g": lambda x: x >= 0,
        "product_length_cm": lambda x: x >= 0,
        "product_height_cm": lambda x: x >= 0,
        "product_width_cm": lambda x: x >= 0,
        "product_photos_qty": lambda x: x >= 0
    }
}


# ==========================================================
# 2. SUSPICIOUS CATEGORICAL VALUES
# ==========================================================

suspicious_words = {
    "unknown",
    "undefined",
    "not_defined",
    "n/a",
    "na",
    "null",
    "none",
    "?"
}


# ==========================================================
# 3. CLEAN + AUDIT ALL TABLES
# ==========================================================

cleaned_tables = {}
audit = []


for table_name, original_df in tables.items():

    df = original_df.copy()


    # ------------------------------------------------------
    # A. REMOVE EXACT DUPLICATES
    # ------------------------------------------------------

    duplicates_removed = int(df.duplicated().sum())

    df = (
        df
        .drop_duplicates()
        .reset_index(drop=True)
    )


    # ------------------------------------------------------
    # B. CONVERT DATE / TIMESTAMP COLUMNS
    # ------------------------------------------------------

    date_columns = [
        col for col in df.columns
        if "date" in col.lower()
        or "timestamp" in col.lower()
    ]

    converted_dates = []

    for col in date_columns:

        old_dtype = df[col].dtype

        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )

        if old_dtype != df[col].dtype:
            converted_dates.append(col)


    # ------------------------------------------------------
    # C. MISSING VALUES
    # ------------------------------------------------------

    missing = df.isna().sum()

    missing = missing[missing > 0].to_dict()


    # ------------------------------------------------------
    # D. BUSINESS RULE VALIDATION
    # ------------------------------------------------------

    business_rule_violations = {}

    if table_name in rules:

        for column, rule in rules[table_name].items():

            if column in df.columns:

                non_null = df[column].dropna()

                valid = rule(non_null)

                invalid_count = int((~valid).sum())

                if invalid_count > 0:

                    business_rule_violations[column] = (
                        invalid_count
                    )


    # ------------------------------------------------------
    # E. SUSPICIOUS CATEGORICAL VALUES
    # ------------------------------------------------------

    suspicious_categories = {}

    categorical_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns

    for col in categorical_columns:

        values = (
            df[col]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
        )

        suspicious = values[
            values.isin(suspicious_words)
        ]

        if len(suspicious) > 0:

            suspicious_categories[col] = (
                suspicious
                .value_counts()
                .to_dict()
            )


    # ------------------------------------------------------
    # F. SAVE CLEANED TABLE
    # ------------------------------------------------------

    cleaned_tables[table_name] = df


    # ------------------------------------------------------
    # G. SAVE AUDIT RESULT
    # ------------------------------------------------------

    audit.append({

        "table": table_name,

        "original_rows":
            len(original_df),

        "final_rows":
            len(df),

        "duplicates_removed":
            duplicates_removed,

        "date_columns_converted":
            converted_dates,

        "missing_values":
            missing,

        "business_rule_violations":
            business_rule_violations,

        "suspicious_categories":
            suspicious_categories
    })


# ==========================================================
# 4. DISPLAY AUDIT REPORT
# ==========================================================

for report in audit:

    print("\n" + "=" * 75)

    print(
        f"TABLE: {report['table']}"
    )

    print("=" * 75)

    print(
        f"Rows: "
        f"{report['original_rows']:,}"
        f" → "
        f"{report['final_rows']:,}"
    )

    print(
        "Exact duplicates removed:",
        report["duplicates_removed"]
    )

    print(
        "Date columns converted:",
        report["date_columns_converted"]
        if report["date_columns_converted"]
        else "None"
    )

    print(
        "Missing values:",
        report["missing_values"]
        if report["missing_values"]
        else "None"
    )

    print(
        "Business-rule violations:",
        report["business_rule_violations"]
        if report["business_rule_violations"]
        else "None"
    )

    print(
        "Suspicious categories:",
        report["suspicious_categories"]
        if report["suspicious_categories"]
        else "None"
    )


TABLE: olist_orders_dataset
Rows: 99,441 → 99,441
Exact duplicates removed: 0
Date columns converted: ['order_purchase_timestamp', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Missing values: {'order_approved_at': 160, 'order_delivered_carrier_date': 1783, 'order_delivered_customer_date': 2965}
Business-rule violations: None
Suspicious categories: None

TABLE: olist_products_dataset
Rows: 32,951 → 32,951
Exact duplicates removed: 0
Date columns converted: None
Missing values: {'product_category_name': 610, 'product_name_lenght': 610, 'product_description_lenght': 610, 'product_photos_qty': 610, 'product_weight_g': 2, 'product_length_cm': 2, 'product_height_cm': 2, 'product_width_cm': 2}
Business-rule violations: None
Suspicious categories: None

TABLE: olist_sellers_dataset
Rows: 3,095 → 3,095
Exact duplicates removed: 0
Date columns converted: None
Missing values: None
Business-rule violations: None
Suspicious categories: None

TAB

In [17]:
import shutil

zip_path = shutil.make_archive(
    "/content/cleaned_olist",
    "zip",
    "/content/cleaned_olist"
)

print("ZIP created:", zip_path)

ZIP created: /content/cleaned_olist.zip


In [18]:
from google.colab import files

files.download("/content/cleaned_olist.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>